سلول ۱ — Project Path

In [1]:
import os
import sys

# Project root
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:")
print(project_root)

Project root added:
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance


سلول ۲ — Load Settings

In [2]:
from config.settings import *

print("Settings loaded successfully")
print("Processed data path:", DATA_PROCESSED_PATH)
print("Chunks data path:", DATA_CHUNKS_PATH)

Settings loaded successfully
Processed data path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed
Chunks data path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/chunks


سلول ۳ — پیدا کردن فایل‌های Processed

In [3]:
from pathlib import Path
import json

PROCESSED_DIR = Path(DATA_PROCESSED_PATH) / "komeil_roudi"
CHUNKS_DIR = Path(DATA_CHUNKS_PATH) / "komeil_roudi"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

article_files = sorted(PROCESSED_DIR.glob("*/article.json"))

print("Processed directory:", PROCESSED_DIR)
print("Chunks directory:", CHUNKS_DIR)
print("Found processed articles:", len(article_files))

print("\nFirst 5 files:")
for file in article_files[:5]:
    print(file)

Processed directory: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi
Chunks directory: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/chunks/komeil_roudi
Found processed articles: 76

First 5 files:
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi/00D9A87F/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi/0A050EE9/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi/11140315/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi/14509512/article.json
/Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/processed/komeil_roudi/1699F2E4/article.json


سلول ۴ — خواندن یک مقاله متنی برای تست Chunking

In [4]:
sample_article = None
sample_article_id = None

for file in article_files:
    with open(file, "r", encoding="utf-8") as f:
        article = json.load(f)

    if article["content"].strip():
        sample_article = article
        sample_article_id = file.parent.name
        break

print("Sample article ID:", sample_article_id)
print("Title:", sample_article["title"])
print("Content length:", len(sample_article["content"]))

print("\nPreview:")
print(sample_article["content"][:1000])

Sample article ID: 0A050EE9
Title: سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی
Content length: 5731

Preview:
هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و به بیان فنی‌تر، ترکیب شش هنر است. بیان سینمایی یعنی هر یک از آن شش هنر در جای خود و به اندازهٔ خود و با ترکیب مناسب، حامل پیام کارگردان است. با این وصف، در ارزیابی فیلم نمی‌شود بیان سینمایی را نادیده گرفت. مخاطب‌فهمی و مخاطب‌پسندی به معنای عوام‌زدگی یا توده‌گرایی نیست. سینمایی سینماست که فرد را از خانه بیرون بکشد و به پشت درِ گیشه برساند و پولی پرداخت کند و ساعاتی در آنجا

سلول ۵ — تعریف تابع ساده Chunking

فعلاً با تعداد کاراکتر چانک می‌کنیم، نه Token. بعداً در مرحله Embedding دقیق‌ترش می‌کنیم.

In [5]:
def chunk_text(text, chunk_size=1200, chunk_overlap=200):
    """
    Split text into overlapping character-based chunks.
    """

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks

سلول ۶ — تست Chunking روی مقاله نمونه

In [6]:
sample_chunks = chunk_text(
    sample_article["content"],
    chunk_size=1200,
    chunk_overlap=200
)

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3], start=1):
    print("=" * 50)
    print("Chunk:", i)
    print("Length:", len(chunk))
    print()
    print(chunk[:500])

Number of chunks: 6
Chunk: 1
Length: 1200

هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و 
Chunk: 2
Length: 1200

د، بی‌تردید گیشه برای‌شان مهم نیست و سینما نیستند؛ بلکه فیلمی ساختهٔ ارگانی دولتی یا غیردولتی برای ارائهٔ رایگان به مخاطب خود هستند. این جمله برای جانورشناسان خنده‌دار است: «گونه‌ای از لوکسودینا سیکلوتیس را تجسم کنید که بر سر پیل پوزی با رنگدانه‌های پنتون حرکات نوسانی‌اش را تکرار می‌کند.»؛ اما برای ما خنده‌دار نیست؛ چرا که گوینده از ابزاری استفاده کرده که منِ مخاطب به آن دسترسی ندارم. همین جمله را با ابزاری که مخاطب دارد بازنویسی

خروجی درست است، ولی یک مشکل مهم دارد:

Chunkها وسط جمله یا وسط کلمه قطع شده‌اند. برای نسخه اولیه قابل قبول است، اما بهتر است همین الان کمی تمیزترش کنیم.

سلول ۷ — Chunking بهتر با برش روی پایان جمله

In [9]:
def chunk_text_by_sentence(text, chunk_size=1200, chunk_overlap=200):
    """
    Split text into overlapping chunks and try to end each chunk at a sentence boundary.
    """

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)

    sentence_endings = [".", "!", "؟", "؛", "\n"]

    while start < text_length:
        end = min(start + chunk_size, text_length)

        if end < text_length:
            best_end = -1

            for sep in sentence_endings:
                pos = text.rfind(sep, start, end)
                if pos > best_end:
                    best_end = pos

            if best_end != -1 and best_end > start + int(chunk_size * 0.5):
                end = best_end + 1

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == text_length:
            break

        start = max(end - chunk_overlap, 0)

    return chunks

سلول ۸ — تست Chunking بهتر روی مقاله نمونه

In [10]:
sample_chunks = chunk_text_by_sentence(
    sample_article["content"],
    chunk_size=1200,
    chunk_overlap=200
)

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks[:3], start=1):
    print("=" * 50)
    print("Chunk:", i)
    print("Length:", len(chunk))
    print()
    print(chunk[:500])
    print("\n--- End preview ---")
    print(chunk[-300:])

Number of chunks: 6
Chunk: 1
Length: 1132

هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و 

--- End preview ---
 برساند و پولی پرداخت کند و ساعاتی در آنجا بنشیند و به خانه بازگردد. این فرایند نیازمند جذابیت و کشش فیلم است. فیلم‌هایی که ساخته می‌شوند اما پسند مخاطب را در خود ندارند، بی‌تردید گیشه برای‌شان مهم نیست و سینما نیستند؛ بلکه فیلمی ساختهٔ ارگانی دولتی یا غیردولتی برای ارائهٔ رایگان به مخاطب خود هستند.
Chunk: 2
Length: 1117

فیلم است. فیلم‌هایی که ساخته می‌شوند اما پسند مخاطب را در خود ندارند، بی‌تردید گیشه برای‌شان مهم نیست و سینما ن

سلول ۹ — ساخت Chunk با Metadata

In [11]:
def create_article_chunks(article, article_id, chunk_size=1200, chunk_overlap=200):
    """
    Create structured chunks with metadata for one article.
    """

    text_chunks = chunk_text_by_sentence(
        article["content"],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = []

    for index, chunk_text in enumerate(text_chunks, start=1):
        chunk = {
            "chunk_id": f"{article_id}_{index}",
            "article_id": article_id,
            "chunk_index": index,
            "title": article["title"],
            "url": article["url"],
            "text": chunk_text,
            "char_count": len(chunk_text)
        }

        chunks.append(chunk)

    return chunks

سلول ۱۰ — تست ساخت Chunkهای ساختاریافته

In [12]:
sample_structured_chunks = create_article_chunks(
    article=sample_article,
    article_id=sample_article_id,
    chunk_size=1200,
    chunk_overlap=200
)

print("Number of structured chunks:", len(sample_structured_chunks))

print("\nFirst chunk keys:")
print(sample_structured_chunks[0].keys())

print("\nFirst chunk:")
print(json.dumps(sample_structured_chunks[0], ensure_ascii=False, indent=2)[:2000])

Number of structured chunks: 6

First chunk keys:
dict_keys(['chunk_id', 'article_id', 'chunk_index', 'title', 'url', 'text', 'char_count'])

First chunk:
{
  "chunk_id": "0A050EE9_1",
  "article_id": "0A050EE9",
  "chunk_index": 1,
  "title": "سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی",
  "url": "https://www.fintelligence.ir/Blog/Detail/0A050EE9/%D8%B3%D9%87-%D9%85%DB%8C%D8%B2%D8%A7%D9%86-%D8%A7%D8%B5%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%B1%D8%B2%DB%8C%D8%A7%D8%A8%DB%8C-%D9%81%DB%8C%D9%84%D9%85%D9%87%D8%A7%DB%8C-%D8%B3%DB%8C%D9%86%D9%85%D8%A7%DB%8C%DB%8C-%D8%A7%D8%B2-%D9%85%D9%86%D8%B8%D8%B1-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C",
  "text": "هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این م

سلول ۱۱ — Chunk کردن کل مقاله‌ها و ذخیره خروجی

In [13]:
from tqdm import tqdm

all_chunks = []
skipped_articles = 0

for file in tqdm(article_files):
    article_id = file.parent.name

    with open(file, "r", encoding="utf-8") as f:
        article = json.load(f)

    if not article["content"].strip():
        skipped_articles += 1
        continue

    article_chunks = create_article_chunks(
        article=article,
        article_id=article_id,
        chunk_size=1200,
        chunk_overlap=200
    )

    all_chunks.extend(article_chunks)

output_file = CHUNKS_DIR / "chunks.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Total chunks:", len(all_chunks))
print("Skipped empty articles:", skipped_articles)
print("Saved to:", output_file)

100%|██████████| 76/76 [00:00<00:00, 3215.58it/s]

Total chunks: 168
Skipped empty articles: 17
Saved to: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/chunks/komeil_roudi/chunks.json


سلول ۱۲ — بررسی فایل نهایی chunks.json

In [14]:
with open(CHUNKS_DIR / "chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Loaded chunks:", len(chunks))

print("\nFirst chunk keys:")
print(chunks[0].keys())

print("\nFirst chunk:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1500])

print("\nLast chunk:")
print(json.dumps(chunks[-1], ensure_ascii=False, indent=2)[:1500])

Loaded chunks: 168

First chunk keys:
dict_keys(['chunk_id', 'article_id', 'chunk_index', 'title', 'url', 'text', 'char_count'])

First chunk:
{
  "chunk_id": "0A050EE9_1",
  "article_id": "0A050EE9",
  "chunk_index": 1,
  "title": "سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی",
  "url": "https://www.fintelligence.ir/Blog/Detail/0A050EE9/%D8%B3%D9%87-%D9%85%DB%8C%D8%B2%D8%A7%D9%86-%D8%A7%D8%B5%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%B1%D8%B2%DB%8C%D8%A7%D8%A8%DB%8C-%D9%81%DB%8C%D9%84%D9%85%D9%87%D8%A7%DB%8C-%D8%B3%DB%8C%D9%86%D9%85%D8%A7%DB%8C%DB%8C-%D8%A7%D8%B2-%D9%85%D9%86%D8%B8%D8%B1-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C",
  "text": "هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول